<a href="https://colab.research.google.com/github/Sikaaaa3/lab-4-llm-decision-support/blob/main/Lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Student Name: Nortey Dankwah Janet
 Student ID: 23112028

PART 0

In [13]:
#Testing
from google.colab import userdata

API_KEY = userdata.get("GROQ_API_KEY")

print("API key loaded:", API_KEY is not None)

from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "llama-3.1-8b-instant"
print("Client ready.")

API key loaded: True
Client ready.



Section 1 — Talking to an LLM Programmatically

PART 1.1 - First API Call

In [14]:
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    return response.choices[0].message.content

#Asking Question
answer = ask_llm("What is a loan?")
print(answer)


#Printing response for token usage
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is a loan?"},
    ],
    temperature=0.7,
    max_tokens=500,
)

print(response.usage)








A loan is a financial agreement where one party, the lender, provides a sum of money to another party, the borrower, with the expectation that the borrower will repay the loan, usually with interest. The borrower agrees to make regular payments, known as installments or repayments, over a specified period, known as the loan term.

There are many types of loans, including:

1. **Personal loan**: A loan for personal expenses, such as debt consolidation, home improvement, or weddings.
2. **Mortgage loan**: A loan to purchase or refinance a home or property.
3. **Car loan**: A loan to purchase or lease a vehicle.
4. **Student loan**: A loan to finance education expenses.
5. **Business loan**: A loan to finance business operations, expansion, or debt consolidation.
6. **Payday loan**: A short-term loan with high interest rates, often used to cover unexpected expenses.

When you take out a loan, you typically agree to:

1. **Repay the principal**: The original amount borrowed.
2. **Pay inter

Student Reasoning

1.The system role gives the model its overall instructions and behavior. For example, our system prompt could say “You are a helpful assistant.” The user role contains the actual request or information we want the model to respond to, such as “What is a loan?” In our API call, the system message sets the behavior while the user message asks the question.

2.A token is a small piece of text that a language model processes; it can be part of a word, a whole short word, or punctuation. API providers bill by tokens because different requests can contain very different amounts of text and require different amounts of computation. In our example, the call used 363 tokens in total: 46 prompt tokens and 317 completion tokens. So charging per token is more representative of the amount of text and processing used than charging the same amount for every request.

PART 1.2 - Temperature: Randomness Dail

In [15]:
question = "Suggest a name for a savings product for market traders in Accra."

for temperature in [0.0, 1.2]:
    print(f"\n Temperature: {temperature} ")

    for i in range(5):
        answer = ask_llm(question, temperature=temperature)
        print(f"{i+1}. {answer}")


 Temperature: 0.0 
1. Considering the target market of market traders in Accra, I would suggest the following name for a savings product:

1. **Makola Savings**: "Makola" is a popular market in Accra, and using its name would help the product connect with the local market traders.
2. **TradeSafe**: This name emphasizes the safety and security of the savings product, which is essential for market traders who need to manage their finances effectively.
3. **MarketMoola**: "Moola" is a Ghanaian slang for money, and "Market" clearly indicates the product's target audience.
4. **AccraSavings**: This name is straightforward and emphasizes the product's connection to the city of Accra.
5. **PesaPesa**: "Pesa" is a Swahili word for money, and "PesaPesa" means "money money" in Twi, a local language in Ghana. This name could appeal to market traders who are familiar with local languages.

Choose the one that best fits your product's brand identity and messaging.
2. Considering the target market 

Student Reasoning

At temperature 0.0, the model produced very similar and consistent responses across the runs. At temperature 1.2, the responses were much more varied, with different names and suggestions being generated. For the loan decision-support system, temperature 0 is more appropriate because the task requires consistent, predictable extraction and decision-support information rather than creative variation. Higher temperatures would make the outputs less reliable and harder to reproduce.


Section 2 — The Dataset: Loan Application Letters

In [16]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")


6 letters loaded.



Section 3 — Prompt Engineering for the Decision Support System

Part 3.1 — Component 1: Summarization

In [17]:
#Summary of L002 and L006
SUMMARY_PROMPT_V1 = "Summarise this"
V1_L002=ask_llm(f"{SUMMARY_PROMPT_V1}\n{LETTERS['L002']}")
V1_L006=ask_llm(f"{SUMMARY_PROMPT_V1}\n{LETTERS['L006']}")

#Summary using proper prompt
SUMMARY_PROMPT_V2 = """
You are an assistant to a microfinance loan officer.
Be factual and neutral.
Use only information provided in the letter.
Do not invent or assume any details.
Summarize the loan application in 3-4 sentences.
"""
V2_L002=ask_llm(f"{SUMMARY_PROMPT_V2}\n{LETTERS['L002']}", temperature=0)
V2_L006=ask_llm(f"{SUMMARY_PROMPT_V2}\n{LETTERS['L006']}", temperature=0)

print("v1")
print("L002:")
print(V1_L002)
print("\nL006:")
print(V1_L006)

print("\nv2")
print("L002:")
print(V2_L002)
print("\nL006:")
print(V2_L006)










v1
L002:
Kwame Boateng, a commercial driver in Kumasi, is in need of GHS 25,000 to repair his trotro engine and settle personal debts. Due to slow business, he's requesting a loan with the promise of paying it back once his business picks up after the festive season, despite having no collateral at the moment.

L006:
Kofi expressed interest in a loan of GHS 50,000 to start three businesses: a car washing business, a provision shop, and importing phones from Dubai. Despite not having any experience or collateral, he claims to be trustworthy and promises to repay the loan within one year, when the businesses are successful.

v2
L002:
Here is a summary of the loan application:

Kwame Boateng, a commercial driver in Kumasi, is requesting a loan of GHS 25,000 to repair his trotro engine and settle personal debts. He states that business has been slow but expects it to improve after the festive season. He is unable to provide collateral at this time and plans to repay the loan when funds bec

# Comparing outputs of V1 and V2

V1 gave shorter, more general summaries, while V2 was more structured and detailed. V2 clearly included important details such as the loan purpose, repayment period, and collateral status. However, V2 still included claims such as Kofi being “trustworthy,” which is only his own statement and should not be treated as a verified fact. Overall, V2 is more useful for a loan officer because its instructions produce a more focused and factual summary

Student Reasoning

1.V1 was already fairly accurate, but V2 was more structured and explicit. For example, V1 says Kofi “claims to be trustworthy and promises to repay the loan within one year”, while V2 clearly states that he is “22-year-old” and “has no prior experience in any of these ventures.” V2 also makes the lack of collateral clearer by stating “No collateral is offered as security for the loan.” For L002, V2 clearly states that Kwame “expects business to improve after the festive season” and plans to repay “when funds become available.”

2.“No invented details” is essential because loan officers need to make decisions using only the facts provided by the applicant. If the model creates information that is not in the letter, it could lead to an incorrect loan assessment. This failure mode is called hallucination, where an LLM generates information that is not supported by the given source.

Part 3.2 — Component 2: Structured extraction (JSON)

In [18]:
# Prompt definition
EXTRACT_PROMPT = """
You are an assistant extracting information from loan applications.

Return ONLY a JSON object with exactly these keys:

{
  "applicant_name": "string",
  "amount_ghs": number,
  "purpose": "string",
  "monthly_profit_ghs": number or null,
  "has_collateral_or_guarantor": true or false,
  "repayment_months": number or null
}

Rules:
- Use only information stated in the letter.
- If a field is not stated, use null.
- Do not guess or invent information.
- Return only valid JSON.

Example:

Letter:
"My name is Ama. I need GHS 5,000 to buy a sewing machine.
I make GHS 600 profit monthly and can repay in 10 months.
I have no collateral."

JSON:
{
  "applicant_name": "Ama",
  "amount_ghs": 5000,
  "purpose": "buy a sewing machine",
  "monthly_profit_ghs": 600,
  "has_collateral_or_guarantor": false,
  "repayment_months": 10
}
"""

#Extract_fields function
import json
def extract_fields(letter_text, temperature=0):
  try:
    response=ask_llm(f"{EXTRACT_PROMPT}\nLetter:\n{letter_text}",
                     temperature=temperature)
    response=response.strip()

    if response.startswith("```json"):
      response=response[7:]

    if response.endswith("```"):
      response=response[:-3]

    return json.loads(response.strip())
  except Exception as e:
    print("Warning: Could not parse the model response")

    return None
#Running on all six letters
import pandas as pd

results = []

for letter_id, letter_text in LETTERS.items():
  print(f"\nProcessing {letter_id}...")
  extracted = extract_fields(letter_text)

  if extracted is not None:
      extracted["letter_id"] = letter_id
      results.append(extracted)
      print("SUCCESS")
  else:
      print("FAILED")

df = pd.DataFrame(results)

display(df)






Processing L001...
SUCCESS

Processing L002...
SUCCESS

Processing L003...
SUCCESS

Processing L004...
SUCCESS

Processing L005...
SUCCESS

Processing L006...
SUCCESS


,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months,letter_id
0,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0,L001
1,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN,L002
2,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0,L003
3,Yaw Owusu,12000,feed and 500 new layers for poultry farm,1500.0,True,18.0,L004
4,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0,L005
5,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0,L006


Student Reasoning

1.The few-shot example should not come from the six letters because it could cause the model to copy information from the letters instead of learning how to extract the fields correctly. This could make the evaluation less reliable.

2.The instruction is important because some information is not provided in the letters. In our extraction results, for example, L002, L005, and L006 have NaN for monthly profit because the letters did not state a monthly profit. Using null prevents the model from inventing a value that was not given.

3.Temperature 0 is appropriate for extraction because we want the model to give consistent and predictable values from the letters. Our reliability test showed this: at temperature 0, all 5/5 outputs were valid and identical. For creative tasks, however, variation is useful, so a higher temperature can produce more diverse ideas and responses.

Part 3.3 — Component 3: The decision-support brief

In [19]:
BRIEF_PROMPT = """
You are an assistant supporting a microfinance loan officer.

Review the loan application letter and the extracted JSON information.

Produce a decision-support brief with exactly these sections:

1. Strengths
- Give bullet points based only on information in the letter.

2. Risks / Red Flags
- Give bullet points identifying concerns or uncertainties.

3. Missing Information
- List information or documents the loan officer should request.

4. Suggested Next Step
- Suggest a practical next step such as requesting documents,
  inviting the applicant for an interview, or flagging for senior review.

Important:
- Be factual and neutral.
- Use only information provided in the letter and extracted JSON.
- Do not invent or assume details.
- Do NOT approve or reject the loan.
- The final loan decision must be made by a human.
"""

def generate_brief(letter_text, extracted):
  prompt= f"""
{BRIEF_PROMPT}

Letter:
{letter_text}

Extracted:
{extracted}
"""

  return ask_llm(prompt, temperature=0)

print("L001 ")
print(generate_brief(LETTERS["L001"], extract_fields(LETTERS["L001"])))

print("\nL003 ")
print(generate_brief(LETTERS["L003"], extract_fields(LETTERS["L003"])))

print("\n L006 ")
print(generate_brief(LETTERS["L006"], extract_fields(LETTERS["L006"])))








L001 
**Decision-Support Brief**

**1. Strengths:**

* The applicant, Akosua Mensah, has 12 years of experience selling provisions at Makola Market.
* She has a stable source of income, with a monthly profit of GHS 900.
* She has a history of saving with the susu scheme, having saved GHS 2,500 over two years without missing any contributions.
* She has a clear repayment plan, committing to repay GHS 450 monthly over 20 months.
* She has a guarantor, her sister, a teacher.

**2. Risks / Red Flags:**

* The loan amount is GHS 8,000, which is a significant amount for a small business owner.
* The applicant's business is expanding into a new product line (frozen foods), which may carry additional risks.
* There is no information about the applicant's business expenses, profit margins, or cash flow.
* The guarantor's financial situation and ability to cover the loan in case of default are not disclosed.

**3. Missing Information:**

* Business expenses and profit margins
* Cash flow project

Student Reasoning

1. Yes, the system identified the main strengths and red flags in both applications. L003 had strong evidence such as a registered business, monthly profit, sales records, and a fixed deposit that could serve as collateral. L006, on the other hand, had major red flags such as no business experience, no collateral or guarantor, no existing businesses, and no clear monthly profit. The briefs therefore showed the important differences between the strong and weak applications.

2. We forbade the model from outputting “approve” or “reject” because the AI is meant to support the loan officer rather than make the final decision. Practically, this prevents the system from making a final decision when important information may be missing. Ethically, it keeps humans accountable and reduces the risk of unfairly denying an applicant based only on an AI-generated assessment.

Part 3.4 — Commit your prompt templates

In [20]:
%%writefile prompts.py

SUMMARY_PROMPT = """
You are an assistant to a microfinance loan officer.
Summarize the loan application in 3-4 sentences.
Be factual and neutral.
Use only information provided in the letter.
Do not invent or assume any details.
"""

EXTRACT_PROMPT = """
You are an assistant extracting information from loan applications.

Return ONLY a JSON object with exactly these keys:
applicant_name, amount_ghs, purpose, monthly_profit_ghs,
has_collateral_or_guarantor, repayment_months

If a field is not stated in the letter, use null.
Do not guess or invent information.
Return only valid JSON.
"""

BRIEF_PROMPT = """
You are an assistant supporting a microfinance loan officer.

Produce a decision-support brief with:
1. Strengths
2. Risks / Red Flags
3. Missing Information
4. Suggested Next Step

Be factual and neutral. Use only information provided.
Do not invent or assume details.
Do NOT approve or reject the loan.
The final loan decision must be made by a human.
"""

Overwriting prompts.py


In [21]:
from google.colab import files

files.download("prompts.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Commit hash: [246c520]


Section 4 — Evaluation: Quality, Reliability, Appropriateness


Part 4.1 — Extraction accuracy against gold labels

In [25]:
#Comparing fields
fields = [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]

rows = []

for field in fields:
    row = {"field": field}
    correct_count = 0

    for letter_id in ["L001", "L003", "L006"]:
        actual = df.loc[df["letter_id"] == letter_id, field].iloc[0]
        expected = GOLD[letter_id][field]

        # Names are case-insensitive
        if field == "applicant_name":
            correct = str(actual).lower() == str(expected).lower()
        else:
            correct = actual == expected

        row[letter_id] = "✓" if correct else "✗"

        if correct:
            correct_count += 1


    #Calculating accuracy
    row["accuracy"] = f"{correct_count}/3"
    rows.append(row)
#Table display
accuracy_df = pd.DataFrame(rows)

display(accuracy_df)

,field,L001,L003,L006,accuracy
0,applicant_name,✓,✓,✓,3/3
1,amount_ghs,✓,✓,✓,3/3
2,purpose,✗,✗,✗,0/3
3,monthly_profit_ghs,✓,✓,✗,2/3
4,has_collateral_or_guarantor,✓,✓,✓,3/3
5,repayment_months,✓,✓,✓,3/3


Part 4.2 — Reliability: is the system consistent?

In [28]:
# Running at temp=0.0
result =[]
for i in range(5):
  output =extract_fields(LETTERS["L004"], temperature = 0.0)
  result.append(output)
print(result)

#Running at temp = 1.0
result_1 =[]
for i in range(5):
  output_1 =extract_fields(LETTERS["L004"], temperature = 1.0)
  result_1.append(output_1)
print(result_1)

import json
valid_0 = [r for r in result if r is not None]
valid_1 = [r for r in result_1 if r is not None]

print("Temperature 0 valid JSON:", len(valid_0), "/5")
print("Temperature 1.0 valid JSON:", len(valid_1), "/5")

unique_0 = set(json.dumps(r, sort_keys=True) for r in valid_0)
unique_1 = set(json.dumps(r, sort_keys=True) for r in valid_1)

print("Temperature 0 unique outputs:", len(unique_0))
print("Temperature 1.0 unique outputs:", len(unique_1))

[{'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'feed and 500 new layers for poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}, {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'feed and 500 new layers for poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}, {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'feed and 500 new layers for poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}, {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'feed and 500 new layers for poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}, {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'feed and 500 new layers for poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}]
[None, {'

Part 4.3 — Hallucination probing

In [30]:
test1 = ask_llm(
    f"{SUMMARY_PROMPT_V2}\n\nWhat is the applicant's credit score?\n\nLetter:\n{LETTERS['L001']}",
    temperature=0
)

print(test1)

irrelevant_text = """
Today's weather in Accra will be sunny with temperatures around 30°C.
There may be some rain later in the evening.
"""

test2 = extract_fields(irrelevant_text, temperature=0)

print(test2)

The loan application is from Akosua Mensah, a 12-year vendor at Makola Market, who is requesting a GHS 8,000 loan to purchase a deep freezer and expand her business into frozen foods. She has a savings history with the organization, having saved GHS 2,500 over two years through the susu scheme. Akosua proposes a repayment plan of GHS 450 monthly over 20 months, with her sister acting as a guarantor.
{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}


In [ ]:
Student Reasoning

1. The system performed well on most fields, getting 3/3 for applicant name, amount, collateral/guarantor, and repayment months. Monthly profit had 2/3 accuracy, while purpose had 0/3, making it the hardest field. The purpose field was difficult because the model often captured the correct meaning but reworded it, so it did not exactly match the gold-standard value.

2. At temperature 0,all 5 runs produced valid JSON and there was only 1 unique output, meaning the results were identical. At temperature 1.0, only 4/5 runs produced valid JSON, with 4 unique outputs among them. This shows that lower temperature provides more consistent results, which is important in production systems where reliable and repeatable extraction is needed

3.The system did not hallucinate in our two tests. When asked for a credit score that was not in the letter, the summarizer did not invent one. When given irrelevant weather information, the extractor returned None or all fields instead of creating an applicant. The risk could be further reduced by instructing the model to use only information stated in the letter, use null when information is missing, and never guess, combined with human review before important decisions
  
    
  

    
  


Part 4.4 — Appropriateness: should this system exist?

1.If the bank fully automated the decisions, applicants like L002 and L006 could be unfairly rejected because their applications contain weaknesses such as poor repayment information or lack of collateral. Also, applicants who write poor English but operate successful businesses could be disadvantaged because the system may judge their written application instead of the actual strength of their business.

2.Loan letters contain personal and financial information, so sending them to a third-party API in another country creates privacy and security risks. Before deployment, the institution should check how the API stores and protects the data, who can access it, whether the provider allows the data to be used for other purposes, and whether the arrangement meets applicable data-protection requirements.

3.Two safeguards
Human review:A qualified loan officer must review the AI's output before making the final decision.

Appeal and monitoring:Applicants should be able to challenge decisions, while the institution regularly monitors the system for errors, bias, and unfair treatmen



Section 5 — Reflection

1. Prompting as engineering
Prompt iteration is similar to tuning model hyperparameters because both involve changing something, testing the result, and improving performance. However, prompting changes the instructions given to the model, while hyperparameter tuning changes how the model learns or behaves during training.

2.Trust
I would not trust the system to run completely unattended because the evaluation showed that higher temperature caused inconsistent extraction results. The reliability test in Section 4.2 influenced my answer most because temperature 1.0 produced only 4/5 valid JSON results and different outputs across runs

3.Cost and Scale
One API request used 363 tokens in total (46 prompt tokens and 317 completion tokens). If each application required a similar amount, processing 1,000 applications would require approximately 363,000 tokens per month. This means provider choice is important because token limits and pricing can significantly affect the cost and scalability of the system

4. Calling an API is better for this task because a foundation model already has strong language understanding and does not require us to collect a large dataset and train our own model. Training our own model would make more sense when we have a large, task-specific dataset, need specialized behavior, or need greater control over the model and data